# 1.Importing Libraries

In [48]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

In [49]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HUGGINGFACEHUB_API_TOKEN")

# 2.Embedding

In [50]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# 3.Documents

In [51]:
doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

docs = [doc1, doc2, doc3, doc4, doc5]

# 4.Vector Store

In [52]:
vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="chroma_db",
    collection_name="sample"
)

# 5.Add Documents

In [53]:

vector_store.add_documents(docs)

['348078a6-7b60-4653-a1c7-7acbd03898ea',
 '7e711e7b-4d98-4b4f-9f0d-8b026b9ab68c',
 '186e86ee-f911-42af-be48-1df5f8ffa81a',
 'e854ec2e-ca22-4fc0-9ed4-977186dff5d0',
 '4c9a9bda-b719-40c0-ba06-2b944df932c0']

# 6.View Documents

In [54]:
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['98a87923-addc-4968-9027-94d9238430af',
  '69d1f404-6e73-4056-a3fb-c56507fff9cd',
  'b7ca58fa-04f7-42e0-9b02-dbb561732f42',
  '8aab048c-7644-46c3-bc95-b5407213e190',
  '89790dc7-52c6-4729-9c87-6c91c991d249',
  '348078a6-7b60-4653-a1c7-7acbd03898ea',
  '7e711e7b-4d98-4b4f-9f0d-8b026b9ab68c',
  '186e86ee-f911-42af-be48-1df5f8ffa81a',
  'e854ec2e-ca22-4fc0-9ed4-977186dff5d0',
  '4c9a9bda-b719-40c0-ba06-2b944df932c0'],
 'embeddings': array([[ 0.00994725,  0.06914335, -0.0514712 , ..., -0.0354334 ,
          0.01284813,  0.01248285],
        [ 0.00127746,  0.0312985 , -0.02375378, ..., -0.00518364,
         -0.03280616,  0.02737711],
        [-0.10265916,  0.02650809,  0.02271503, ..., -0.03359751,
         -0.07984945, -0.01507709],
        ...,
        [-0.10265916,  0.02650809,  0.02271503, ..., -0.03359751,
         -0.07984945, -0.01507709],
        [ 0.02123393, -0.0246855 , -0.0449437 , ..., -0.1099581 ,
          0.00572559,  0.09915373],
        [ 0.01873975,  0.04382844, 

# 7.Search Documents

In [55]:
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=1
)


[Document(id='8aab048c-7644-46c3-bc95-b5407213e190', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

In [56]:
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)


[Document(id='8aab048c-7644-46c3-bc95-b5407213e190', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='e854ec2e-ca22-4fc0-9ed4-977186dff5d0', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

# 8. Search Documents with Similarity Score

In [57]:
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='8aab048c-7644-46c3-bc95-b5407213e190', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9693601727485657),
 (Document(id='e854ec2e-ca22-4fc0-9ed4-977186dff5d0', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9693601727485657)]

# 9. Meta-data Filtering

In [58]:
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "Mumbai Indians"}
)

[(Document(id='69d1f404-6e73-4056-a3fb-c56507fff9cd', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  1.8651394844055176),
 (Document(id='7e711e7b-4d98-4b4f-9f0d-8b026b9ab68c', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  1.8651394844055176),
 (Document(id='8aab048c-7644-46c3-bc95-b5407213e190', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  1.9243816137313843),
 (Document(id='e854ec2e-ca22-4fc0-9ed4-977186dff5d0', metadata={'team': 'Mumbai Indians'}, 

# 10.Update Documents

In [59]:
updated_doc1 = Document(
    page_content="Rajat Patidar, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

# 11.View Documents

In [60]:
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['98a87923-addc-4968-9027-94d9238430af',
  '69d1f404-6e73-4056-a3fb-c56507fff9cd',
  'b7ca58fa-04f7-42e0-9b02-dbb561732f42',
  '8aab048c-7644-46c3-bc95-b5407213e190',
  '89790dc7-52c6-4729-9c87-6c91c991d249',
  '348078a6-7b60-4653-a1c7-7acbd03898ea',
  '7e711e7b-4d98-4b4f-9f0d-8b026b9ab68c',
  '186e86ee-f911-42af-be48-1df5f8ffa81a',
  'e854ec2e-ca22-4fc0-9ed4-977186dff5d0',
  '4c9a9bda-b719-40c0-ba06-2b944df932c0'],
 'embeddings': array([[ 0.00994725,  0.06914335, -0.0514712 , ..., -0.0354334 ,
          0.01284813,  0.01248285],
        [ 0.00127746,  0.0312985 , -0.02375378, ..., -0.00518364,
         -0.03280616,  0.02737711],
        [-0.10265916,  0.02650809,  0.02271503, ..., -0.03359751,
         -0.07984945, -0.01507709],
        ...,
        [-0.10265916,  0.02650809,  0.02271503, ..., -0.03359751,
         -0.07984945, -0.01507709],
        [ 0.02123393, -0.0246855 , -0.0449437 , ..., -0.1099581 ,
          0.00572559,  0.09915373],
        [ 0.01873975,  0.04382844, 

# 12.Delete Documents

In [61]:
vector_store.delete(ids=['98a87923-addc-4968-9027-94d9238430af'])

# 13.view Documents

In [62]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['69d1f404-6e73-4056-a3fb-c56507fff9cd',
  'b7ca58fa-04f7-42e0-9b02-dbb561732f42',
  '8aab048c-7644-46c3-bc95-b5407213e190',
  '89790dc7-52c6-4729-9c87-6c91c991d249',
  '348078a6-7b60-4653-a1c7-7acbd03898ea',
  '7e711e7b-4d98-4b4f-9f0d-8b026b9ab68c',
  '186e86ee-f911-42af-be48-1df5f8ffa81a',
  'e854ec2e-ca22-4fc0-9ed4-977186dff5d0',
  '4c9a9bda-b719-40c0-ba06-2b944df932c0'],
 'embeddings': array([[ 0.00127746,  0.0312985 , -0.02375378, ..., -0.00518364,
         -0.03280616,  0.02737711],
        [-0.10265916,  0.02650809,  0.02271503, ..., -0.03359751,
         -0.07984945, -0.01507709],
        [ 0.02123393, -0.0246855 , -0.0449437 , ..., -0.1099581 ,
          0.00572559,  0.09915373],
        ...,
        [-0.10265916,  0.02650809,  0.02271503, ..., -0.03359751,
         -0.07984945, -0.01507709],
        [ 0.02123393, -0.0246855 , -0.0449437 , ..., -0.1099581 ,
          0.00572559,  0.09915373],
        [ 0.01873975,  0.04382844, -0.04304259, ..., -0.07801618,
         -0